In [3]:
import pandas as pd

df_raw = pd.read_excel('dataset_circor.xlsx')  # ajusta la ruta si esta en otro lado
ciclos_por_archivo = df_raw.groupby('archivo').size()

for umbral in [2, 3, 5, 8]:
    archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= umbral]
    print(f"umbral >= {umbral} ciclos: quedan {len(archivos_ok)} de {len(ciclos_por_archivo)} archivos ({100*len(archivos_ok)/len(ciclos_por_archivo):.0f}%)")

umbral >= 2 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 3 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 5 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 8 ciclos: quedan 649 de 1079 archivos (60%)


In [4]:
for umbral in [2, 3, 5, 8]:
    archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= umbral].index
    etiquetas_ok = df_raw[df_raw['archivo'].isin(archivos_ok)].drop_duplicates('archivo')['Etiqueta']
    print(f"umbral >= {umbral}: Sano={sum(etiquetas_ok==0)}  Soplo={sum(etiquetas_ok==2)}")

umbral >= 2: Sano=826  Soplo=253
umbral >= 3: Sano=826  Soplo=253
umbral >= 5: Sano=826  Soplo=253
umbral >= 8: Sano=508  Soplo=141


In [5]:
df_circor = pd.read_excel('dataset_circor.xlsx')  # ajusta la ruta

feature_cols_circor = [c for c in df_circor.columns if c not in ('Etiqueta', 'archivo', 'paciente_id')]
X_circor = df_circor[feature_cols_circor].values
y_circor = df_circor['Etiqueta'].values
grupos_circor = df_circor['paciente_id'].values

print(f"{df_circor.shape[0]} filas de {df_circor['paciente_id'].nunique()} pacientes")
print("\npacientes por clase:")
print(df_circor.groupby('Etiqueta')['paciente_id'].nunique())

11153 filas de 586 pacientes

pacientes por clase:
Etiqueta
0    459
2    127
Name: paciente_id, dtype: int64


In [6]:
sgkf_circor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

# chequeo rapido de fuga (mismo principio de siempre, ahora resumido)
fuga_total = 0
for idx_entrena, idx_prueba in sgkf_circor.split(X_circor, y_circor, groups=grupos_circor):
    pacientes_entrena = set(grupos_circor[idx_entrena])
    pacientes_prueba = set(grupos_circor[idx_prueba])
    fuga_total += len(pacientes_entrena & pacientes_prueba)
print(f"pacientes repetidos entre lados, sumando los 5 folds: {fuga_total}")

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0),
    'Gradient Boosting': GradientBoostingClassifier(random_state=0),
}

for nombre, modelo in modelos.items():
    pipe = Pipeline([('escalador', StandardScaler()), ('clf', modelo)])
    scores = cross_val_score(pipe, X_circor, y_circor, cv=sgkf_circor, groups=grupos_circor, scoring='accuracy')
    print(f"{nombre:22s} exactitud = {scores.mean():.3f} +/- {scores.std():.3f}")

NameError: name 'StratifiedGroupKFold' is not defined

In [5]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, accuracy_score
import joblib

df_v2 = pd.read_excel('dataset_circor_v2.xlsx')
ciclos_por_archivo = df_v2.groupby('archivo').size()
print(ciclos_por_archivo.describe())
print(f"\narchivos con menos de 5 ciclos: {(ciclos_por_archivo < 5).sum()} de {len(ciclos_por_archivo)}")

count    3005.000000
mean       29.715141
std        11.681073
min         1.000000
25%        22.000000
50%        28.000000
75%        37.000000
max        87.000000
dtype: float64

archivos con menos de 5 ciclos: 23 de 3005


In [3]:
df_v2 = pd.read_excel('dataset_circor_v2.xlsx')

ciclos_por_archivo = df_v2.groupby('archivo').size()
archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= 5].index
df_v2 = df_v2[df_v2['archivo'].isin(archivos_ok)]
print(f"tras filtro de calidad: {df_v2.shape[0]} filas, {df_v2['paciente_id'].nunique()} pacientes")

feature_cols_v2 = [c for c in df_v2.columns if c not in ('Etiqueta', 'archivo', 'paciente_id')]
df_v2['paciente_id'] = df_v2['paciente_id'].astype(str)

pacientes_prueba = pd.read_csv('pacientes_prueba_final.csv')['paciente_id'].astype(str)
es_prueba = df_v2['paciente_id'].isin(pacientes_prueba)

df_dev_v2 = df_v2[~es_prueba]
df_prueba_v2 = df_v2[es_prueba]

print(f"dev:           {df_dev_v2.shape[0]} filas, {df_dev_v2['paciente_id'].nunique()} pacientes")
print(f"prueba final:  {df_prueba_v2.shape[0]} filas, {df_prueba_v2['paciente_id'].nunique()} pacientes")

tras filtro de calidad: 89238 filas, 873 pacientes
dev:           76630 filas, 756 pacientes
prueba final:  12608 filas, 117 pacientes


In [6]:
X_dev_v2 = df_dev_v2[feature_cols_v2].values
y_dev_v2 = df_dev_v2['Etiqueta'].values
grupos_dev_v2 = df_dev_v2['paciente_id'].values

sgkf_v2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0),
    'Gradient Boosting': GradientBoostingClassifier(random_state=0),
}

for nombre, modelo in modelos.items():
    pipe = Pipeline([('escalador', StandardScaler()), ('clf', modelo)])
    scores = cross_val_score(pipe, X_dev_v2, y_dev_v2, cv=sgkf_v2, groups=grupos_dev_v2, scoring='accuracy')
    print(f"{nombre:22s} exactitud = {scores.mean():.3f} +/- {scores.std():.3f}")

Regresion Logistica    exactitud = 0.695 +/- 0.016
SVM                    exactitud = 0.738 +/- 0.009
Random Forest          exactitud = 0.841 +/- 0.020
Gradient Boosting      exactitud = 0.843 +/- 0.022
